In [1]:
import numpy as np
from scipy import linalg
import matplotlib.pyplot as plt

Energy computation for single electron system (i.e., Hydrogen)

In [2]:
def hydrogen_energy(n):
    return -1.0 / (2.0 * n**2)
 
print("="*60)
print("HYDROGEN ATOM ENERGY LEVELS")
print("="*60)
for n in range(1, 6):
    E_hartree = hydrogen_energy(n)
    E_eV = E_hartree * 27.211
    print(f"n = {n}: E = {E_hartree:.6f} Hartree = {E_eV:.3f} eV")

HYDROGEN ATOM ENERGY LEVELS
n = 1: E = -0.500000 Hartree = -13.605 eV
n = 2: E = -0.125000 Hartree = -3.401 eV
n = 3: E = -0.055556 Hartree = -1.512 eV
n = 4: E = -0.031250 Hartree = -0.850 eV
n = 5: E = -0.020000 Hartree = -0.544 eV


Many Electron Problem

In [ ]:
"""
For N electrons, the wavefunction depends on 3N spatial + N spin coordinates, the electron-electron repulsion term (in Hamiltonian) couples all electrons, making the problem non-separable.
"""
 
def count_determinants(n_electrons, n_orbitals):
    """
    Count the number of Slater determinants in FCI.
    
    For N electrons in M orbitals:
    - N/2 alpha electrons can be placed in M orbitals: C(M, N/2) ways
    - N/2 beta electrons can be placed in M orbitals: C(M, N/2) ways
    - Total: C(M, N/2)² determinants
    
    This is the dimension of the FCI matrix!
    """
    from math import comb
    n_alpha = n_electrons // 2
    n_beta = n_electrons - n_alpha
    return comb(n_orbitals, n_alpha) * comb(n_orbitals, n_beta)
 
print("\n" + "="*60)
print("THE COMBINATORIAL EXPLOSION")
print("="*60)
print("\nNumber of Slater determinants for different systems:\n")
 
systems = [
    ("H₂ (2e, 4 orbitals)", 2, 4),
    ("H₂O (10e, 14 orbitals)", 10, 14),
    ("Benzene (30e, 36 orbitals)", 30, 36),
    ("Fe₂S₂ cluster (50e, 50 orbitals)", 50, 50),
    ("[4Fe-4S] cluster (100e, 76 orbitals)", 100, 76),
]
 
for name, n_e, n_orb in systems:
    n_det = count_determinants(n_e, n_orb)
    print(f"{name:40s}: {n_det:>20,.0f} determinants")
 
print("\n>>> The [4Fe-4S] cluster is Prof. Sharma's benchmark system!")
print(">>> Classical methods CANNOT solve this exactly.")
print(">>> This is why SHCI, DMRG, and AFQMC are essential.")

In [ ]:
"""
A Slater determinant is an antisymmetrized product of spin-orbitals:
 
|Φ⟩ = |χ₁χ₂...χₙ⟩ = 1/√N! |χ₁(1) χ₂(1) ... χₙ(1)|
                         |χ₁(2) χ₂(2) ... χₙ(2)|
                         |  ⋮      ⋮    ⋱    ⋮   |
                         |χ₁(N) χ₂(N) ... χₙ(N)|
 
Properties:
1. Antisymmetric under particle exchange (Pauli principle)
2. Normalized if orbitals are orthonormal
3. Not an eigenfunction of Ĥ in general (except for 1 electron)
"""
 
def slater_determinant_example():
    """
    Demonstrate Slater determinant properties with 2 electrons.
    
    For 2 electrons in orbitals φ₁ and φ₂:
    |Φ⟩ = 1/√2 [φ₁(1)φ₂(2) - φ₁(2)φ₂(1)]
    
    Using matrix form:
    |Φ⟩ = 1/√2! det|φ₁(1) φ₂(1)|
                   |φ₁(2) φ₂(2)|
    """
    print("\n" + "="*60)
    print("SLATER DETERMINANT EXAMPLE: 2 electrons")
    print("="*60)
    
    # Represent orbitals as vectors on a grid
    x = np.linspace(-3, 3, 100)
    
    # Simple Gaussian orbitals (like 1s)
    phi_1 = np.exp(-x**2)        # Orbital 1 centered at origin
    phi_2 = np.exp(-(x-1)**2)    # Orbital 2 centered at x=1
    
    # Normalize
    phi_1 = phi_1 / np.sqrt(np.trapz(phi_1**2, x))
    phi_2 = phi_2 / np.sqrt(np.trapz(phi_2**2, x))
    
    # Overlap
    S_12 = np.trapz(phi_1 * phi_2, x)
    print(f"Overlap ⟨φ₁|φ₂⟩ = {S_12:.4f}")
    print("(Non-zero overlap means these are NOT orthogonal)")
    
    # For a proper Slater determinant, we need orthonormal orbitals
    # Let's orthogonalize using Gram-Schmidt
    phi_2_orth = phi_2 - S_12 * phi_1
    phi_2_orth = phi_2_orth / np.sqrt(np.trapz(phi_2_orth**2, x))
    
    S_12_new = np.trapz(phi_1 * phi_2_orth, x)
    print(f"After orthogonalization: ⟨φ₁|φ₂'⟩ = {S_12_new:.2e}")
    
    return x, phi_1, phi_2_orth
 
x, phi_1, phi_2 = slater_determinant_example()